In [ ]:
import shutil
import os

from rds_chat_analysis import NOTEBOOK_DIR
from rds_chat_analysis.client import init_session
from syft_core.config import CONFIG_PATH_ENV
from syft_core import Client as SyftboxClient

In [ ]:
RDS_DO_CONFIG = "./.rds/wildchat/data_owner_config.json"
RDS_DS_CONFIG = "./.rds/wildchat/data_scientist_config.json"

In [ ]:
# IMPORTANT - Set env to load the correct syftbox config
# If you have a single syftbox set up on your system and you want to use that for this notebook, this step is not necessary.
os.environ[CONFIG_PATH_ENV] = RDS_DO_CONFIG

In [ ]:
# Directory we're saving all non-syftbox data to (e.g. configuration, staging folders, execution artifacts, etc.)
WORK_DIR = NOTEBOOK_DIR / "v2"

In [ ]:
do_syftbox_client = SyftboxClient.load()

# DO connects to it's own RDS app
do_client = init_session(host=do_syftbox_client.email)

# Test if connection is working
health_check = do_client.rpc.health()
print(f"Health check: {health_check}")
print(f"Logged in on RDS admin client: {do_client.is_admin}")

# DO creates dataset

In [ ]:
# Create local staging folders for the mock and private data
DATASET_NAME = "Wildchat-postgres"

staging_data_dir = WORK_DIR / "staging" / DATASET_NAME
private_dir = staging_data_dir / "private"
mock_dir = staging_data_dir / "mock"
markdown_path = staging_data_dir / "README.md"

shutil.rmtree(staging_data_dir, ignore_errors=True)
private_dir.mkdir(parents=True, exist_ok=True)
mock_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Copy the mock and private credentials to the local staging folders
MOCK_CREDENTIALS = WORK_DIR / "config_mock.toml"
PRIVATE_CREDENTIALS = WORK_DIR / "config_private.toml"

_ = shutil.copy(MOCK_CREDENTIALS, mock_dir / "config.toml")
_ = shutil.copy(PRIVATE_CREDENTIALS, private_dir / "config.toml")

print(f"Mock dir structure: {mock_dir}")
for file in mock_dir.iterdir():
    print(f"└──📄 {file.name}")
print(f"Private dir structure: {private_dir}")
for file in private_dir.iterdir():
    print(f"└──📄 {file.name}")

In [ ]:
# Create a markdown description for the dataset
description_markdown = """
# Wildchat Postgres Dataset
"""

markdown_path.write_text(description_markdown.strip())

In [ ]:
# Upload the dataset to the RDS app
dataset_exists = len(do_client.dataset.get_all(name=DATASET_NAME)) > 0
if dataset_exists:
    print(f"Retrieving existing dataset: {DATASET_NAME}")
    wildchat_dataset = do_client.dataset.get(name=DATASET_NAME)
else:
    print(f"Creating dataset: {DATASET_NAME}")
    wildchat_dataset = do_client.dataset.create(
        name=DATASET_NAME,
        path=private_dir,
        mock_path=mock_dir,
        summary="A embedded wildchat dataset in postgres.",
        description_path=markdown_path,
    )

In [ ]:
# Check if the dataset was created successfully
wildchat_dataset = do_client.dataset.get(name=DATASET_NAME)
wildchat_dataset.describe()

# DO configures custom functions [TODO]

Custom functions are currently pre-defined for this project, and manual configuring is not supported yet.

Proposal on what this feature would look like: https://docs.google.com/document/d/1JZy5trYTS6T89tjdGYawNJdmfvVIH3Uyj5h_HErp-DE/edit?usp=sharing

### Current pre-defined functions

- [rds_chat_analysis.job_functions.py](../../src/rds_chat_analysis/job_functions.py) contains all pre-configured functions.

# DO reviews and executes

Before executing the cells below, first run the next notebook to submit a job as data scientist.

In [ ]:
pending_jobs = do_client.job.get_all(status="pending_code_review")
pending_jobs

In [ ]:
job = pending_jobs[0]

do_client.review_job(job)

In [ ]:
# Run on the private dataset
do_client.run_private(job)

# DO reviews and shares the results

In [ ]:
import json

job_results = do_client.job.review_results(job)
job_results.describe()

for k, v in job_results.outputs.items():
    print(f"Contents of {k}: {json.dumps(v, indent=2)}")

In [ ]:
# If we're happy nothing sensitive is in the results, we can share them to the data scientist

do_client.jobs.share_results(job)